In [ ]:
!pip install facenet-pytorch tqdm scikit-learn opencv-python pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 9

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import shutil

from tqdm import tqdm
from facenet_pytorch import MTCNN
from sklearn.model_selection import train_test_split

In [ ]:
BASE_DIR = "/content/drive/MyDrive/deepfake_project"

RAW_DATA_DIR = os.path.join(BASE_DIR, "data/raw_videos")
FRAME_DIR = os.path.join(BASE_DIR, "data/extracted_frames")
FACE_DIR = os.path.join(BASE_DIR, "data/cropped_faces")
META_DIR = os.path.join(BASE_DIR, "data/metadata")


os.makedirs(FRAME_DIR, exist_ok=True)
os.makedirs(FACE_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

classes = ["real", "fake"]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device: cpu


Did run extract_frames loop previous session, stored them in drive in /content/drive/MyDrive/deepfake_project/data/extracted_frames. But lost connection so the cell outputs are lost.

In [ ]:
# DO NOT RUN, DONE
def extract_frames(video_path, output_dir, frame_skip=5):

    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    frame_count = 0
    saved_count = 0

    while True:

        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_skip == 0:

            frame_path = os.path.join(
                output_dir,
                f"frame_{saved_count:04d}.jpg"
            )

            cv2.imwrite(frame_path, frame)
            saved_count += 1

        frame_count += 1

    cap.release()

In [ ]:
# DO NOT RUN, DONE

for label in classes:

    class_root = os.path.join(RAW_DATA_DIR, label)

    for method in os.listdir(class_root):

        method_path = os.path.join(class_root, method)

        if not os.path.isdir(method_path):
            continue

        for compression in os.listdir(method_path):

            compression_path = os.path.join(method_path, compression)

            videos_path = os.path.join(compression_path, "videos")

            if not os.path.exists(videos_path):
                continue

            videos = os.listdir(videos_path)

            for video_name in tqdm(videos, desc=f"{label}-{method}-{compression}"):

                if not video_name.endswith(".mp4"):
                    continue

                video_path = os.path.join(videos_path, video_name)

                save_dir = os.path.join(
                    FRAME_DIR,
                    label,
                    video_name.split(".")[0]
                )

                extract_frames(video_path, save_dir, frame_skip=5)

In [ ]:
mtcnn = MTCNN(
    image_size=224,
    margin=20,
    device=device,
    keep_all=False
)

In [ ]:
# For face cropping
MAX_FRAMES_PER_VIDEO = 30   # LIMIT OUTPUT, might reduce overfitting to longer videos and prevent data inbalance
FRAME_SKIP = 3             # SAMPLE EVERY 3 FRAMES

In [ ]:
done_file = "processed_videos.txt"

if os.path.exists(done_file):
    with open(done_file, "r") as f:
        done_videos = set(f.read().splitlines())
else:
    done_videos = set()

In [ ]:
# cpu safe
def crop_face(image_path, save_path):
    try:
        if os.path.exists(save_path):
            return True

        img = cv2.imread(image_path)
        if img is None:
            return False

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        face = mtcnn(img_rgb)

        if face is None:
            return False

        face = face.permute(1, 2, 0).cpu().numpy()
        face = ((face + 1) / 2 * 255).astype(np.uint8)

        face_bgr = cv2.cvtColor(face, cv2.COLOR_RGB2BGR)
        cv2.imwrite(save_path, face_bgr)

        return True

    except:
        return False

In [ ]:
# resume safe pipeline
for label in classes:

    frame_root = os.path.join(FRAME_DIR, label)
    videos = sorted(os.listdir(frame_root))

    for video in tqdm(videos, desc=f"Cropping {label}"):

        video_id = f"{label}/{video}"

        # SKIP IF ALREADY DONE
        if video_id in done_videos:
            continue

        video_frame_dir = os.path.join(frame_root, video)
        if not os.path.isdir(video_frame_dir):
            continue

        output_dir = os.path.join(FACE_DIR, label, video)
        os.makedirs(output_dir, exist_ok=True)

        frames = sorted(os.listdir(video_frame_dir))

        saved = 0

        for i, frame_name in enumerate(frames):

            if i % FRAME_SKIP != 0:
                continue

            if saved >= MAX_FRAMES_PER_VIDEO:
                break

            frame_path = os.path.join(video_frame_dir, frame_name)
            save_path = os.path.join(output_dir, frame_name)

            if crop_face(frame_path, save_path):
                saved += 1

        # MARK VIDEO DONE
        with open(done_file, "a") as f:
            f.write(video_id + "\n")

Cropping fake: 100%|██████████| 175/175 [59:42<00:00, 20.47s/it]


In [ ]:
data = []

label_map = {"real": 0, "fake": 1}

for label_name in classes:

    label_dir = os.path.join(FACE_DIR, label_name)

    if not os.path.exists(label_dir):
        continue

    videos = os.listdir(label_dir)

    for video in videos:

        video_dir = os.path.join(label_dir, video)

        if not os.path.isdir(video_dir):
            continue

        frames = os.listdir(video_dir)

        for frame_name in frames:

            frame_path = os.path.join(video_dir, frame_name)

            data.append({
                "frame_path": frame_path,
                "video_id": f"{label_name}/{video}",
                "label": label_map[label_name]
            })

df = pd.DataFrame(data)

print(df.head())
print("Total frames:", len(df))

                                          frame_path  video_id  label
0  /content/drive/MyDrive/deepfake_project/data/c...  real/000      0
1  /content/drive/MyDrive/deepfake_project/data/c...  real/000      0
2  /content/drive/MyDrive/deepfake_project/data/c...  real/000      0
3  /content/drive/MyDrive/deepfake_project/data/c...  real/000      0
4  /content/drive/MyDrive/deepfake_project/data/c...  real/000      0
Total frames: 32058


In [ ]:
meta_path = os.path.join(META_DIR, "frame_metadata.csv")

df.to_csv(meta_path, index=False)

print("Saved to:", meta_path)

Saved to: /content/drive/MyDrive/deepfake_project/data/metadata/frame_metadata.csv


In [ ]:
unique_videos = df["video_id"].unique()

train_videos, test_videos = train_test_split(
    unique_videos,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_df = df[df["video_id"].isin(train_videos)].reset_index(drop=True)
test_df = df[df["video_id"].isin(test_videos)].reset_index(drop=True)

print("Train frames:", len(train_df))
print("Test frames:", len(test_df))
print("Train videos:", len(train_videos))
print("Test videos:", len(test_videos))

Train frames: 25696
Test frames: 6362
Train videos: 937
Test videos: 235


In [ ]:
train_path = os.path.join(META_DIR, "train.csv")
test_path = os.path.join(META_DIR, "test.csv")

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("Saved train to:", train_path)
print("Saved test to:", test_path)

Saved train to: /content/drive/MyDrive/deepfake_project/data/metadata/train.csv
Saved test to: /content/drive/MyDrive/deepfake_project/data/metadata/test.csv


In [ ]:
import os

RAW_REAL = "/content/drive/MyDrive/deepfake_project/data/raw_videos/real"
RAW_FAKE = "/content/drive/MyDrive/deepfake_project/data/raw_videos/fake"

def count_videos(root):

    total = 0

    for method in os.listdir(root):

        method_path = os.path.join(root, method)
        if not os.path.isdir(method_path):
            continue

        for comp in os.listdir(method_path):

            comp_path = os.path.join(method_path, comp)
            videos_path = os.path.join(comp_path, "videos")

            if not os.path.exists(videos_path):
                continue

            videos = [v for v in os.listdir(videos_path) if v.endswith(".mp4")]

            total += len(videos)

    return total

real_count = count_videos(RAW_REAL)
fake_count = count_videos(RAW_FAKE)

print("Real videos:", real_count)
print("Fake videos:", fake_count)
print("TOTAL raw videos:", real_count + fake_count)

Real videos: 1000
Fake videos: 1000
TOTAL raw videos: 2000


In [ ]:
df["label"].value_counts()

,count
label,
0,27345
1,4713


The dataset exhibits class imbalance due to preprocessing and face detection variability. To mitigate this, we will use class-weighted loss and maybe balanced sampling (to ensure each batch has equal real/fake data) during training.

Below, there is the initial crop_face function and loop. But due to collab's gpu time limit, the process was interrupted. (The process of cropping 'real' frames was 87% done before gpu usage was exceeded.) Therefore I implemented a new resume-safe crop_face function above to save time.

In [ ]:
# fixed crop func
def crop_face(image_path, save_path):

    try:
        img = cv2.imread(image_path)

        if img is None:
            print(f"Could not read: {image_path}")
            return False

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        face = mtcnn(img_rgb)

        if face is None:
            print(f"No face detected: {image_path}")
            return False

        # CHW -> HWC
        face = face.permute(1, 2, 0).cpu().numpy()

        # DENORMALIZE [-1,1] -> [0,255]
        face = ((face + 1) / 2 * 255).astype(np.uint8)

        # RGB -> BGR
        face_bgr = cv2.cvtColor(face, cv2.COLOR_RGB2BGR)

        cv2.imwrite(save_path, face_bgr)

        return True

    except Exception as e:
        print(f"Error processing {image_path}")
        print(e)
        return False

In [ ]:
for label in classes:

    frame_root = os.path.join(FRAME_DIR, label)

    videos = os.listdir(frame_root)

    for video in tqdm(videos, desc=f"Cropping {label}"):

        video_frame_dir = os.path.join(frame_root, video)

        if not os.path.isdir(video_frame_dir):
            continue

        output_dir = os.path.join(FACE_DIR, label, video)
        os.makedirs(output_dir, exist_ok=True)

        frames = sorted(os.listdir(video_frame_dir))

        saved = 0

        for i, frame_name in enumerate(frames):

            # SKIP FRAMES
            if i % FRAME_SKIP != 0:
                continue

            if saved >= MAX_FRAMES_PER_VIDEO:
                break

            frame_path = os.path.join(video_frame_dir, frame_name)
            save_path = os.path.join(output_dir, frame_name)

            success = crop_face(frame_path, save_path)

            if success:
              saved += 1

Cropping real:  87%|████████▋ | 871/1000 [2:12:23<20:58,  9.75s/it]